# CNN-GRU Electricity Price Forecasting
### Day-Ahead (24hr) + Real-Time (1hr) Models

In [ ]:
import numpy as np
import pandas as pd
import datetime
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import seaborn as sns
import math
import os
import copy

import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.nn.functional as F
from tqdm import tqdm
from sklearn.preprocessing import MinMaxScaler

## 1. Load & Inspect Data

In [ ]:
energy_weather = pd.read_csv('../../data/processed/energy_weather_merged.csv')
energy_weather.info()
energy_weather.head()

## 2. Feature Engineering & Column Definitions

In [ ]:
# ── Target columns ────────────────────────────────────────────────────────────
# Day-ahead model predicts: price day ahead  (24 hrs ahead)
# Real-time model predicts:  price actual    (1 hr  ahead)
target_col_dayahead  = 'price day ahead'
target_col_realtime  = 'price actual'

all_target_cols = [
    'gen_solar',
    'gen_wind',
    'total load actual',
    'price actual',
    'price day ahead',
]

# ── Forecast columns — kept for real-time model inputs ───────────────────────
# These represent officially published day-ahead forecasts (e.g. from ENTSO-E).
# They are what a market participant actually has available at inference time.
# NOT dropped anymore — used as Stream-2 inputs to the real-time model.
forecast_cols = [
    'forecast solar day ahead',
    'forecast wind onshore day ahead',
    'total load forecast',
]

# ── Real-time forecast feature set ───────────────────────────────────────────
# At real-time inference we have: published day-ahead forecasts + settled
# day-ahead price (day-ahead price is always known before real-time delivery).
realtime_forecast_cols = forecast_cols + ['price day ahead']

# ── Identifier / non-numeric columns ─────────────────────────────────────────
identifier_cols = ['time', 'date', 'city_name', 'hour']

In [ ]:
# ── Datetime conversion ───────────────────────────────────────────────────────
energy_weather['time'] = pd.to_datetime(energy_weather['time'], utc=True)
energy_weather['date'] = pd.to_datetime(energy_weather['date'])
energy_weather = energy_weather.sort_values('time').reset_index(drop=True)

# ── Cyclical time features ────────────────────────────────────────────────────
energy_weather['hour_sin']    = np.sin(2 * np.pi * energy_weather['hour'] / 24)
energy_weather['hour_cos']    = np.cos(2 * np.pi * energy_weather['hour'] / 24)

energy_weather['day_of_week'] = energy_weather['date'].dt.dayofweek
energy_weather['day_sin']     = np.sin(2 * np.pi * energy_weather['day_of_week'] / 7)
energy_weather['day_cos']     = np.cos(2 * np.pi * energy_weather['day_of_week'] / 7)

energy_weather['month']       = energy_weather['date'].dt.month
energy_weather['month_sin']   = np.sin(2 * np.pi * energy_weather['month'] / 12)
energy_weather['month_cos']   = np.cos(2 * np.pi * energy_weather['month'] / 12)

energy_weather['is_weekend']  = (energy_weather['day_of_week'] >= 5).astype(int)

# ── Calendar year for fold splitting ─────────────────────────────────────────
energy_weather['year'] = energy_weather['date'].dt.year

# ── Lag features (for all targets) ───────────────────────────────────────────
# NOTE: with WINDOW_SIZE=168 the model already sees 1 week of raw history,
# so lags are supplementary explicit signals rather than the primary memory.
lag_hours = [1, 2, 3, 24, 48]

for target in all_target_cols:
    col = target.replace(' ', '_')
    for lag in lag_hours:
        energy_weather[f'{col}_lag_{lag}'] = energy_weather[target].shift(lag)

energy_weather = energy_weather.dropna().reset_index(drop=True)
print(f"Shape after feature engineering: {energy_weather.shape}")

In [ ]:
sin_cos_cols = [
    'hour_sin', 'hour_cos',
    'day_sin',  'day_cos',
    'month_sin','month_cos',
    'is_weekend',
]

lag_cols = [
    f"{t.replace(' ', '_')}_lag_{lag}"
    for t in all_target_cols
    for lag in lag_hours
]

# Historical feature columns fed into the CNN-GRU encoder (Stream 1)
hist_feature_cols = [
    # Generation
    'gen_hydro', 'gen_pumped_hydro', 'gen_fossil', 'gen_nuclear', 'gen_other',
    # Weather
    'temp_Barcelona', 'temp_Bilbao', 'temp_Madrid', 'temp_Seville', 'temp_Valencia',
    'pressure_Barcelona', 'pressure_Bilbao', 'pressure_Madrid', 'pressure_Seville', 'pressure_Valencia',
    'humidity_Barcelona', 'humidity_Bilbao', 'humidity_Madrid', 'humidity_Seville', 'humidity_Valencia',
    'wind_speed_Barcelona', 'wind_speed_Bilbao', 'wind_speed_Madrid', 'wind_speed_Seville', 'wind_speed_Valencia',
    'wind_deg_Barcelona', 'wind_deg_Bilbao', 'wind_deg_Madrid', 'wind_deg_Seville', 'wind_deg_Valencia',
    'rain_1h_Barcelona', 'rain_1h_Bilbao', 'rain_1h_Madrid', 'rain_1h_Seville', 'rain_1h_Valencia',
    'rain_3h_Barcelona', 'rain_3h_Bilbao', 'rain_3h_Madrid', 'rain_3h_Seville', 'rain_3h_Valencia',
    'snow_3h_Barcelona', 'snow_3h_Bilbao', 'snow_3h_Madrid', 'snow_3h_Seville', 'snow_3h_Valencia',
    'clouds_all_Barcelona', 'clouds_all_Bilbao', 'clouds_all_Madrid', 'clouds_all_Seville', 'clouds_all_Valencia',
    'temp_avg',
    # Cyclical (pre-bounded [-1,1], not scaled)
] + sin_cos_cols + lag_cols

# Columns that need MinMaxScaler (exclude sin/cos which are already bounded)
hist_scale_cols   = [c for c in hist_feature_cols if c not in sin_cos_cols]
hist_noscale_cols = sin_cos_cols

print(f"Historical feature columns : {len(hist_feature_cols)}")
print(f"  → to scale               : {len(hist_scale_cols)}")
print(f"  → not scaled (sin/cos)   : {len(hist_noscale_cols)}")
print(f"\nReal-time forecast columns : {len(realtime_forecast_cols)}")
print(realtime_forecast_cols)

## 3. Year-Aligned Walk-Forward Cross-Validation

Splits on calendar year boundaries (2 train / 1 val / 1 test) so the model always trains on complete seasonal cycles and folds never cut through the middle of a year.

In [ ]:
all_years = sorted(energy_weather['year'].unique())
print(f"Available years: {all_years}")

TRAIN_YEARS = 2
VAL_YEARS   = 1
TEST_YEARS  = 1
FOLD_STEP   = 1   # advance window by 1 year per fold

folds = []
fold_num = 1

for start in range(0, len(all_years) - TRAIN_YEARS - VAL_YEARS - TEST_YEARS + 1, FOLD_STEP):
    train_yrs = all_years[start                         : start + TRAIN_YEARS]
    val_yrs   = all_years[start + TRAIN_YEARS           : start + TRAIN_YEARS + VAL_YEARS]
    test_yrs  = all_years[start + TRAIN_YEARS + VAL_YEARS : start + TRAIN_YEARS + VAL_YEARS + TEST_YEARS]

    df_train = energy_weather[energy_weather['year'].isin(train_yrs)].copy()
    df_val   = energy_weather[energy_weather['year'].isin(val_yrs)].copy()
    df_test  = energy_weather[energy_weather['year'].isin(test_yrs)].copy()

    # ── Scale historical features (fit on train only) ─────────────────────────
    scaler_X = MinMaxScaler()
    scaler_X.fit(df_train[hist_scale_cols])

    X_train_sc = scaler_X.transform(df_train[hist_scale_cols])
    X_val_sc   = scaler_X.transform(df_val[hist_scale_cols])
    X_test_sc  = scaler_X.transform(df_test[hist_scale_cols])

    X_train = np.hstack([X_train_sc, df_train[hist_noscale_cols].values])
    X_val   = np.hstack([X_val_sc,   df_val[hist_noscale_cols].values])
    X_test  = np.hstack([X_test_sc,  df_test[hist_noscale_cols].values])

    # ── Scale real-time forecast features (fit on train only) ─────────────────
    scaler_fc = MinMaxScaler()
    scaler_fc.fit(df_train[realtime_forecast_cols])

    Xfc_train = scaler_fc.transform(df_train[realtime_forecast_cols])
    Xfc_val   = scaler_fc.transform(df_val[realtime_forecast_cols])
    Xfc_test  = scaler_fc.transform(df_test[realtime_forecast_cols])

    # ── Scale targets (one scaler per target, fit on train only) ──────────────
    target_scalers = {}
    y_train_dict, y_val_dict, y_test_dict = {}, {}, {}

    for target in all_target_cols:
        sc = MinMaxScaler()
        sc.fit(df_train[[target]])
        y_train_dict[target]   = sc.transform(df_train[[target]])
        y_val_dict[target]     = sc.transform(df_val[[target]])
        y_test_dict[target]    = sc.transform(df_test[[target]])
        target_scalers[target] = sc

    folds.append({
        'fold'            : fold_num,
        'train_years'     : train_yrs,
        'val_years'       : val_yrs,
        'test_years'      : test_yrs,
        # Historical features (encoder input)
        'X_train'         : X_train,
        'X_val'           : X_val,
        'X_test'          : X_test,
        # Real-time forecast features (dense head input)
        'Xfc_train'       : Xfc_train,
        'Xfc_val'         : Xfc_val,
        'Xfc_test'        : Xfc_test,
        # Targets
        'y_train'         : y_train_dict,
        'y_val'           : y_val_dict,
        'y_test'          : y_test_dict,
        # Scalers
        'scaler_X'        : scaler_X,
        'scaler_forecast' : scaler_fc,
        'target_scalers'  : target_scalers,
        # Timestamps
        'train_dates'     : df_train['time'].values,
        'val_dates'       : df_val['time'].values,
        'test_dates'      : df_test['time'].values,
    })

    print(f"\nFold {fold_num}:")
    print(f"  Train : {train_yrs}  ({len(df_train):,} rows)")
    print(f"  Val   : {val_yrs}   ({len(df_val):,} rows)")
    print(f"  Test  : {test_yrs}   ({len(df_test):,} rows)")
    fold_num += 1

print(f"\n{len(folds)} fold(s) created")

In [ ]:
fig, ax = plt.subplots(figsize=(14, 3 + len(folds)))
colors  = {'Train': 'steelblue', 'Val': 'orange', 'Test': 'green'}

for fold_data in folds:
    f = fold_data['fold']
    train_min = pd.Timestamp(fold_data['train_dates'].min())
    train_max = pd.Timestamp(fold_data['train_dates'].max())
    val_min   = pd.Timestamp(fold_data['val_dates'].min())
    val_max   = pd.Timestamp(fold_data['val_dates'].max())
    test_min  = pd.Timestamp(fold_data['test_dates'].min())
    test_max  = pd.Timestamp(fold_data['test_dates'].max())

    for left, right, color, label in [
        (train_min, train_max, 'steelblue', 'Train'),
        (val_min,   val_max,   'orange',    'Val'),
        (test_min,  test_max,  'green',     'Test'),
    ]:
        width = (right - left).days
        ax.barh(f, width, left=left, color=color, alpha=0.7, height=0.4)
        ax.text(left, f, f' {label}', va='center', fontsize=8,
                color='white', fontweight='bold')

ax.xaxis_date()
fig.autofmt_xdate()
ax.set_xlabel('Date')
ax.set_ylabel('Fold')
ax.set_title('Year-Aligned Walk-Forward CV — 2yr Train / 1yr Val / 1yr Test')
patches = [mpatches.Patch(color=c, label=l) for l, c in colors.items()]
ax.legend(handles=patches, loc='lower right')
plt.tight_layout()
plt.show()

## 4. Global Settings & Device

In [ ]:
device = torch.device(
    'mps'  if torch.backends.mps.is_available() else
    'cuda' if torch.cuda.is_available()          else
    'cpu'
)
print(f"Device: {device}")

# ── Sequence / horizon ────────────────────────────────────────────────────────
WINDOW_SIZE = 168   # 1 week of hourly history fed into encoder
HORIZON_DA  = 24    # day-ahead model predicts next 24 hours
HORIZON_RT  = 1     # real-time model predicts next 1 hour

# ── Batch / loader ────────────────────────────────────────────────────────────
BATCH_SIZE   = 256 if (torch.cuda.is_available() or torch.backends.mps.is_available()) else 64
NUM_WORKERS  = 4   if  torch.cuda.is_available() else 0
PIN_MEMORY   = torch.cuda.is_available()

# ── Model ──────────────────────────────────────────────────────────────────────
INPUT_SIZE_HIST = folds[0]['X_train'].shape[1]
INPUT_SIZE_FC   = folds[0]['Xfc_train'].shape[1]   # n_forecast_features for RT model
HIDDEN_SIZE     = 256
NUM_LAYERS      = 2
DROPOUT         = 0.3

print(f"\nWindow size         : {WINDOW_SIZE} hrs")
print(f"Day-ahead horizon   : {HORIZON_DA} hrs")
print(f"Real-time horizon   : {HORIZON_RT} hr")
print(f"Batch size          : {BATCH_SIZE}")
print(f"Hist feature size   : {INPUT_SIZE_HIST}")
print(f"Forecast feat size  : {INPUT_SIZE_FC}")

## 5. Dataset Classes

Two separate datasets — one per model. Both use a 168-hour (1 week) lookback window.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Dataset A: Day-Ahead — predicts next 24 hourly prices
# Input  : last 168 hours of historical features
# Target : price day ahead for each of the next 24 hours
# ─────────────────────────────────────────────────────────────────────────────
class DayAheadDataset(Dataset):
    def __init__(self, X_hist, y, window_size=168, horizon=24):
        self.X    = torch.FloatTensor(X_hist)
        self.y    = torch.FloatTensor(y).squeeze(-1)
        self.win  = window_size
        self.hor  = horizon

    def __len__(self):
        return len(self.X) - self.win - self.hor + 1

    def __getitem__(self, idx):
        x_hist     = self.X[idx : idx + self.win]              # (168, n_hist)
        y_next_day = self.y[idx + self.win : idx + self.win + self.hor]  # (24,)
        return x_hist, y_next_day


# ─────────────────────────────────────────────────────────────────────────────
# Dataset B: Real-Time — predicts next single hour price
# Input  : last 168 hours of historical features  (Stream 1)
#        + published forecast features for target hour  (Stream 2)
# Target : price actual at next hour
#
# Stream 2 uses PUBLISHED day-ahead forecasts (from ENTSO-E / TSOs):
#   forecast solar day ahead, forecast wind onshore day ahead,
#   total load forecast, price day ahead
# These are always available before real-time delivery — no data leakage.
# ─────────────────────────────────────────────────────────────────────────────
class RealTimeDataset(Dataset):
    def __init__(self, X_hist, X_forecast, y, window_size=168):
        self.X_hist     = torch.FloatTensor(X_hist)
        self.X_forecast = torch.FloatTensor(X_forecast)
        self.y          = torch.FloatTensor(y).squeeze(-1)
        self.win        = window_size

    def __len__(self):
        return len(self.X_hist) - self.win

    def __getitem__(self, idx):
        x_hist       = self.X_hist[idx : idx + self.win]    # (168, n_hist)
        x_forecast_t = self.X_forecast[idx + self.win]      # (n_fc_features,)
        y_t          = self.y[idx + self.win]                # scalar
        return x_hist, x_forecast_t, y_t

In [ ]:
def make_loaders_dayahead(fold_data, target,
                           window_size=WINDOW_SIZE, horizon=HORIZON_DA,
                           batch_size=BATCH_SIZE, num_workers=NUM_WORKERS,
                           pin_memory=PIN_MEMORY):
    shared = dict(batch_size=batch_size, shuffle=False,
                  num_workers=num_workers, pin_memory=pin_memory,
                  persistent_workers=(num_workers > 0))
    tr = DataLoader(
        DayAheadDataset(fold_data['X_train'], fold_data['y_train'][target],
                        window_size, horizon),
        drop_last=True, **shared)
    va = DataLoader(
        DayAheadDataset(fold_data['X_val'], fold_data['y_val'][target],
                        window_size, horizon),
        drop_last=False, **shared)
    te = DataLoader(
        DayAheadDataset(fold_data['X_test'], fold_data['y_test'][target],
                        window_size, horizon),
        drop_last=False, **shared)
    return tr, va, te


def make_loaders_realtime(fold_data, target,
                           window_size=WINDOW_SIZE,
                           batch_size=BATCH_SIZE, num_workers=NUM_WORKERS,
                           pin_memory=PIN_MEMORY):
    shared = dict(batch_size=batch_size, shuffle=False,
                  num_workers=num_workers, pin_memory=pin_memory,
                  persistent_workers=(num_workers > 0))
    tr = DataLoader(
        RealTimeDataset(fold_data['X_train'], fold_data['Xfc_train'],
                        fold_data['y_train'][target], window_size),
        drop_last=True, **shared)
    va = DataLoader(
        RealTimeDataset(fold_data['X_val'], fold_data['Xfc_val'],
                        fold_data['y_val'][target], window_size),
        drop_last=False, **shared)
    te = DataLoader(
        RealTimeDataset(fold_data['X_test'], fold_data['Xfc_test'],
                        fold_data['y_test'][target], window_size),
        drop_last=False, **shared)
    return tr, va, te


# ── Build all loaders ─────────────────────────────────────────────────────────
all_loaders = {}   # all_loaders[fold_num][model_type][target]

for fold_data in folds:
    fn = fold_data['fold']
    all_loaders[fn] = {'dayahead': {}, 'realtime': {}}

    # Day-ahead: target is 'price day ahead'
    tr, va, te = make_loaders_dayahead(fold_data, target_col_dayahead)
    all_loaders[fn]['dayahead'][target_col_dayahead] = {'train': tr, 'val': va, 'test': te}

    # Real-time: target is 'price actual'
    tr, va, te = make_loaders_realtime(fold_data, target_col_realtime)
    all_loaders[fn]['realtime'][target_col_realtime] = {'train': tr, 'val': va, 'test': te}

    print(f"Fold {fn} loaders ready")

# ── Quick shape verification ──────────────────────────────────────────────────
fold1 = folds[0]
fn    = fold1['fold']

print("\n── Day-Ahead loader shapes ──")
x, y = next(iter(all_loaders[fn]['dayahead'][target_col_dayahead]['train']))
print(f"  X      : {tuple(x.shape)}   → (batch, 168, n_hist_features)")
print(f"  y      : {tuple(y.shape)}   → (batch, 24)")

print("\n── Real-Time loader shapes ──")
x, xfc, y = next(iter(all_loaders[fn]['realtime'][target_col_realtime]['train']))
print(f"  X_hist : {tuple(x.shape)}   → (batch, 168, n_hist_features)")
print(f"  X_fc   : {tuple(xfc.shape)} → (batch, n_forecast_features)")
print(f"  y      : {tuple(y.shape)}   → (batch,)")

## 6. Model Architecture

Both models share the same **CNN encoder** and **bidirectional GRU + attention** backbone. They differ only in their output heads:
- **Day-Ahead**: autoregressive decoder → 24 hourly predictions
- **Real-Time**: dense head with injected forecast features → 1 prediction

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Shared building blocks
# ─────────────────────────────────────────────────────────────────────────────

class CNNEncoder(nn.Module):
    """
    Three-layer 1-D CNN that extracts local temporal patterns from the
    168-hour input window.  BatchNorm replaces MaxPool so the full
    sequence length is preserved for the GRU.
    """
    def __init__(self, input_size, cnn_channels=(64, 128, 128)):
        super().__init__()
        self.out_channels = cnn_channels[-1]
        self.features = nn.Sequential(
            nn.Conv1d(input_size,       cnn_channels[0], kernel_size=3, padding=1),
            nn.BatchNorm1d(cnn_channels[0]),
            nn.ReLU(),
            nn.Conv1d(cnn_channels[0],  cnn_channels[1], kernel_size=5, padding=2),
            nn.BatchNorm1d(cnn_channels[1]),
            nn.ReLU(),
            nn.Conv1d(cnn_channels[1],  cnn_channels[2], kernel_size=7, padding=3),
            nn.BatchNorm1d(cnn_channels[2]),
            nn.ReLU(),
        )

    def forward(self, x):
        # x: (batch, seq_len, input_size)
        x = x.permute(0, 2, 1)    # → (batch, input_size, seq_len) for Conv1d
        x = self.features(x)       # → (batch, 128, seq_len)
        x = x.permute(0, 2, 1)    # → (batch, seq_len, 128)       for GRU
        return x


class AttentionLayer(nn.Module):
    """
    Soft attention over GRU hidden states.
    hidden_size must equal the GRU's actual output dimension.
    For a bidirectional GRU with hidden_size H, pass hidden_size=H*2 here.
    """
    def __init__(self, hidden_size):
        super().__init__()
        self.attn = nn.Linear(hidden_size, 1)

    def forward(self, gru_out):
        # gru_out: (batch, seq_len, hidden_size)
        scores  = self.attn(gru_out).squeeze(-1)          # (batch, seq_len)
        weights = torch.softmax(scores, dim=1)             # (batch, seq_len)
        context = torch.bmm(weights.unsqueeze(1), gru_out).squeeze(1)
        # context: (batch, hidden_size)
        return context, weights


class SharedEncoder(nn.Module):
    """
    CNN → Bidirectional GRU → Attention.
    Output: context vector of shape (batch, hidden_size * 2).
    Both the day-ahead and real-time models use this identical encoder.
    """
    def __init__(self, input_size,
                 cnn_channels=(64, 128, 128),
                 hidden_size=256, num_layers=2, dropout=0.3):
        super().__init__()
        self.hidden_size = hidden_size
        self.cnn = CNNEncoder(input_size, cnn_channels)
        self.gru = nn.GRU(
            input_size=cnn_channels[-1],
            hidden_size=hidden_size,
            num_layers=num_layers,
            dropout=dropout if num_layers > 1 else 0.0,
            batch_first=True,
            bidirectional=True,          # output dim = hidden_size * 2
        )
        self.dropout  = nn.Dropout(dropout)
        # AttentionLayer receives hidden_size*2 because GRU is bidirectional
        self.attention = AttentionLayer(hidden_size * 2)

    def forward(self, x_hist):
        # x_hist: (batch, 168, input_size)
        cnn_out           = self.cnn(x_hist)           # (batch, 168, 128)
        gru_out, _        = self.gru(cnn_out)          # (batch, 168, hidden*2)
        gru_out           = self.dropout(gru_out)
        context, attn_wts = self.attention(gru_out)    # (batch, hidden*2)
        return context, attn_wts


# ─────────────────────────────────────────────────────────────────────────────
# Model A: Day-Ahead Price Forecasting
# Predicts: price day ahead  for each of the next 24 hours
# Decoder:  autoregressive GRU — each step consumes its own prior output
# ─────────────────────────────────────────────────────────────────────────────

class DayAheadDecoder(nn.Module):
    """
    Autoregressive decoder that unrolls 24 steps.
    Consumes context as the initial hidden state and feeds each
    prediction back as input to the next step.
    """
    def __init__(self, hidden_size, output_steps=24):
        super().__init__()
        self.output_steps = output_steps
        self.gru = nn.GRU(
            input_size=1,
            hidden_size=hidden_size,
            num_layers=1,
            batch_first=True,
        )
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, context):
        # context: (batch, hidden_size)
        h0        = context.unsqueeze(0)                  # (1, batch, hidden)
        batch     = context.size(0)
        dec_in    = torch.zeros(batch, 1, 1, device=context.device)
        preds     = []

        for _ in range(self.output_steps):
            out, h0 = self.gru(dec_in, h0)               # (batch, 1, hidden)
            pred    = self.fc(out)                         # (batch, 1, 1)
            preds.append(pred.squeeze(-1))                 # (batch, 1)
            dec_in  = pred                                 # feed back

        return torch.cat(preds, dim=1)                    # (batch, 24)


class DayAheadModel(nn.Module):
    """
    Full day-ahead model.
    Input  : x_hist (batch, 168, n_hist_features)
    Output : pred_24hr (batch, 24)  — next-day hourly price day ahead
    """
    def __init__(self, input_size,
                 cnn_channels=(64, 128, 128),
                 hidden_size=256, num_layers=2, dropout=0.3):
        super().__init__()
        self.encoder = SharedEncoder(input_size, cnn_channels,
                                     hidden_size, num_layers, dropout)
        # Decoder receives context of size hidden_size*2 (bidirectional)
        self.decoder = DayAheadDecoder(hidden_size=hidden_size * 2,
                                       output_steps=24)

    def forward(self, x_hist):
        context, attn_wts = self.encoder(x_hist)    # (batch, hidden*2)
        pred_24hr         = self.decoder(context)   # (batch, 24)
        return pred_24hr, attn_wts


# ─────────────────────────────────────────────────────────────────────────────
# Model B: Real-Time Price Forecasting
# Predicts: price actual  for the NEXT SINGLE HOUR
# Head:     dense layer — injects point-in-time published forecast features
#
# Stream 1 (encoder): last 168 hrs of historical actuals
# Stream 2 (head):    published forecast for the target hour
#   [forecast solar day ahead, forecast wind onshore day ahead,
#    total load forecast, price day ahead]
#
# Using published forecasts (not the day-ahead model's outputs) avoids
# cascading errors and mirrors what is genuinely available at inference time.
# ─────────────────────────────────────────────────────────────────────────────

class RealTimeModel(nn.Module):
    """
    Full real-time model.
    Inputs : x_hist      (batch, 168, n_hist_features)
             x_forecast  (batch, n_forecast_features)  ← single target hour
    Output : pred_1hr    (batch, 1)  — next-hour price actual
    """
    def __init__(self, input_size, n_forecast_features,
                 cnn_channels=(64, 128, 128),
                 hidden_size=256, num_layers=2, dropout=0.3):
        super().__init__()
        self.encoder = SharedEncoder(input_size, cnn_channels,
                                     hidden_size, num_layers, dropout)
        # Dense head: context (hidden*2) concatenated with forecast features
        self.fc = nn.Sequential(
            nn.Linear(hidden_size * 2 + n_forecast_features, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 1),
        )

    def forward(self, x_hist, x_forecast):
        # x_hist     : (batch, 168, n_hist)
        # x_forecast : (batch, n_fc_features)
        context, attn_wts = self.encoder(x_hist)              # (batch, hidden*2)
        combined          = torch.cat([context, x_forecast], dim=-1)
        pred_1hr          = self.fc(combined)                  # (batch, 1)
        return pred_1hr, attn_wts

In [ ]:
da_model = DayAheadModel(
    input_size=INPUT_SIZE_HIST,
    hidden_size=HIDDEN_SIZE,
    num_layers=NUM_LAYERS,
    dropout=DROPOUT,
)

rt_model = RealTimeModel(
    input_size=INPUT_SIZE_HIST,
    n_forecast_features=INPUT_SIZE_FC,
    hidden_size=HIDDEN_SIZE,
    num_layers=NUM_LAYERS,
    dropout=DROPOUT,
)

def count_params(m): return sum(p.numel() for p in m.parameters() if p.requires_grad)

print("── Day-Ahead Model ──────────────────────────────────")
print(da_model)
print(f"Trainable params: {count_params(da_model):,}")

print("\n── Real-Time Model ──────────────────────────────────")
print(rt_model)
print(f"Trainable params: {count_params(rt_model):,}")

# ── Quick forward-pass sanity check ──────────────────────────────────────────
fold1  = folds[0]
fn     = fold1['fold']

x_h, y_da = next(iter(all_loaders[fn]['dayahead'][target_col_dayahead]['train']))
x_h2, x_fc, y_rt = next(iter(all_loaders[fn]['realtime'][target_col_realtime]['train']))

with torch.no_grad():
    p_da, attn_da = da_model(x_h)
    p_rt, attn_rt = rt_model(x_h2, x_fc)

print(f"\nDay-ahead output shape : {tuple(p_da.shape)}   → (batch, 24)")
print(f"Attention shape        : {tuple(attn_da.shape)} → (batch, 168)")
print(f"Real-time output shape : {tuple(p_rt.shape)}   → (batch, 1)")
print(f"Attention shape        : {tuple(attn_rt.shape)} → (batch, 168)")

## 7. Loss, Training & Evaluation Functions

In [ ]:
class CustomLoss(nn.Module):
    """MAE + Jensen-Shannon divergence + smoothness penalty."""
    def __init__(self, alpha=0.1, beta=0.01):
        super().__init__()
        self.alpha = alpha
        self.beta  = beta

    def forward(self, y_pred, y_true):
        y_pred = y_pred.squeeze()
        y_true = y_true.squeeze()

        mae  = torch.mean(torch.abs(y_true - y_pred))

        y_ts = torch.softmax(y_true, dim=0)
        y_ps = torch.softmax(y_pred, dim=0)
        m    = 0.5 * (y_ts + y_ps)
        kl1  = torch.sum(y_ts * torch.log(y_ts / (m + 1e-8) + 1e-8))
        kl2  = torch.sum(y_ps * torch.log(y_ps / (m + 1e-8) + 1e-8))
        jsd  = 0.5 * (kl1 + kl2)

        smooth = torch.mean((y_pred[1:] - y_pred[:-1]) ** 2)

        return mae + self.alpha * jsd + self.beta * smooth

In [ ]:
def train_model(model, train_loader, val_loader,
                model_type='dayahead',
                num_epochs=50, lr=1e-3,
                loss_fn='mae', alpha=0.1, beta=0.01):
    """
    Generic training loop for both DayAheadModel and RealTimeModel.

    model_type : 'dayahead'  → unpacks (X, y_24hr); loss over 24 steps
                 'realtime'  → unpacks (X, X_fc, y_1hr); loss over 1 step
    """
    model = model.to(device)

    if loss_fn == 'mae':
        criterion = nn.L1Loss()
    elif loss_fn == 'mse':
        criterion = nn.MSELoss()
    elif loss_fn == 'custom':
        criterion = CustomLoss(alpha=alpha, beta=beta)
    else:
        raise ValueError("loss_fn must be 'mae', 'mse', or 'custom'")

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', patience=5, factor=0.5)

    best_val_loss = float('inf')
    best_weights  = copy.deepcopy(model.state_dict())
    history       = {'train_loss': [], 'val_loss': [], 'lr': []}

    for epoch in range(num_epochs):

        # ── Training ─────────────────────────────────────────────────────────
        model.train()
        train_losses = []
        train_bar    = tqdm(train_loader,
                            desc=f"Ep {epoch+1:>3}/{num_epochs} [Train]",
                            leave=False)

        for batch in train_bar:
            if model_type == 'dayahead':
                X_batch, y_batch = batch
                X_batch = X_batch.to(device)
                y_batch = y_batch.to(device)          # (batch, 24)
                pred, _ = model(X_batch)              # (batch, 24)
                loss    = criterion(pred, y_batch)

            else:  # realtime
                X_batch, Xfc_batch, y_batch = batch
                X_batch   = X_batch.to(device)
                Xfc_batch = Xfc_batch.to(device)
                y_batch   = y_batch.to(device).unsqueeze(-1)  # (batch, 1)
                pred, _   = model(X_batch, Xfc_batch)         # (batch, 1)
                loss      = criterion(pred, y_batch)

            if torch.isnan(loss) or torch.isinf(loss):
                continue

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            train_losses.append(loss.item())
            train_bar.set_postfix(loss=f"{np.mean(train_losses):.4f}")

        avg_train = np.mean(train_losses) if train_losses else float('nan')

        # ── Validation ───────────────────────────────────────────────────────
        model.eval()
        val_losses = []
        val_bar    = tqdm(val_loader,
                          desc=f"Ep {epoch+1:>3}/{num_epochs} [Val]  ",
                          leave=False)

        with torch.no_grad():
            for batch in val_bar:
                if model_type == 'dayahead':
                    X_batch, y_batch = batch
                    X_batch = X_batch.to(device)
                    y_batch = y_batch.to(device)
                    pred, _ = model(X_batch)
                    loss    = criterion(pred, y_batch)

                else:
                    X_batch, Xfc_batch, y_batch = batch
                    X_batch   = X_batch.to(device)
                    Xfc_batch = Xfc_batch.to(device)
                    y_batch   = y_batch.to(device).unsqueeze(-1)
                    pred, _   = model(X_batch, Xfc_batch)
                    loss      = criterion(pred, y_batch)

                if not (torch.isnan(loss) or torch.isinf(loss)):
                    val_losses.append(loss.item())
                    val_bar.set_postfix(loss=f"{np.mean(val_losses):.4f}")

        avg_val = np.mean(val_losses) if val_losses else float('nan')

        if not np.isnan(avg_val) and avg_val < best_val_loss:
            best_val_loss = avg_val
            best_weights  = copy.deepcopy(model.state_dict())
            print(f"  ✅ Ep {epoch+1:>3} — best saved  val={best_val_loss:.4f}")

        if not np.isnan(avg_val):
            scheduler.step(avg_val)

        history['train_loss'].append(avg_train)
        history['val_loss'].append(avg_val)
        history['lr'].append(optimizer.param_groups[0]['lr'])

        print(f"Ep {epoch+1:>3}/{num_epochs} | "
              f"Train: {avg_train:.4f} | Val: {avg_val:.4f} | "
              f"LR: {optimizer.param_groups[0]['lr']:.6f}")

    model.load_state_dict(best_weights)
    print(f"\n✅ Training complete — best val loss: {best_val_loss:.4f}")
    return model, history

In [ ]:
def test_model(model, test_loader, target_scaler, model_type='dayahead'):
    """
    Evaluate on the test set.  Returns inverse-transformed predictions,
    actuals, and a metrics dict.

    model_type : 'dayahead'  → predictions shape (N, 24)
                 'realtime'  → predictions shape (N, 1)
    """
    model.eval()
    criterion = nn.L1Loss()

    all_preds, all_actuals, test_losses = [], [], []
    test_bar = tqdm(test_loader, desc="Testing", leave=True)

    with torch.no_grad():
        for batch in test_bar:
            if model_type == 'dayahead':
                X_batch, y_batch = batch
                X_batch = X_batch.to(device)
                y_batch = y_batch.to(device)
                pred, _ = model(X_batch)
                loss    = criterion(pred, y_batch)

            else:
                X_batch, Xfc_batch, y_batch = batch
                X_batch   = X_batch.to(device)
                Xfc_batch = Xfc_batch.to(device)
                y_batch   = y_batch.to(device).unsqueeze(-1)
                pred, _   = model(X_batch, Xfc_batch)
                loss      = criterion(pred, y_batch)

            test_losses.append(loss.item())
            all_preds.append(pred.cpu().numpy())
            all_actuals.append(y_batch.cpu().numpy())
            test_bar.set_postfix(loss=f"{np.mean(test_losses):.4f}")

    all_preds   = np.concatenate(all_preds,   axis=0)
    all_actuals = np.concatenate(all_actuals, axis=0)

    # Inverse transform
    if model_type == 'dayahead':
        n = all_preds.shape[0]
        preds_orig   = target_scaler.inverse_transform(
            all_preds.reshape(-1, 1)).reshape(n, 24)
        actuals_orig = target_scaler.inverse_transform(
            all_actuals.reshape(-1, 1)).reshape(n, 24)
    else:
        preds_orig   = target_scaler.inverse_transform(
            all_preds.reshape(-1, 1))
        actuals_orig = target_scaler.inverse_transform(
            all_actuals.reshape(-1, 1))

    mae  = np.mean(np.abs(actuals_orig - preds_orig))
    rmse = np.sqrt(np.mean((actuals_orig - preds_orig) ** 2))
    rng  = float(actuals_orig.max() - actuals_orig.min())
    mae_pct = (mae / rng) * 100 if rng > 0 else float('nan')

    print(f"\n===== TEST RESULTS ({model_type}) =====")
    print(f"  MAE      : {mae:.4f}")
    print(f"  RMSE     : {rmse:.4f}")
    print(f"  MAE %    : {mae_pct:.2f}%  (of actual range {rng:.2f})")

    return preds_orig, actuals_orig, {
        'mae': mae, 'rmse': rmse, 'mae_pct': mae_pct, 'range': rng}

In [ ]:
# ── Hyperparameter search using VALIDATION set ────────────────────────────────
# Vary one hyperparameter at a time and compare val loss.
# Do NOT look at test set results during this process.

hp_configs = [
    {'hidden_size': 128, 'dropout': 0.2, 'lr': 1e-3},
    {'hidden_size': 256, 'dropout': 0.3, 'lr': 1e-3},
    {'hidden_size': 256, 'dropout': 0.3, 'lr': 5e-4},
    {'hidden_size': 512, 'dropout': 0.4, 'lr': 1e-3},
]

val_results = []

for cfg in hp_configs:
    print(f"\nConfig: {cfg}")

    # ── Day-ahead ──────────────────────────────────────────────────────────
    da_model = DayAheadModel(
        input_size=INPUT_SIZE_HIST,
        hidden_size=cfg['hidden_size'],
        num_layers=NUM_LAYERS,
        dropout=cfg['dropout'],
    ).to(device)

    da_trained, da_history = train_model(
        da_model,
        all_loaders[fn]['dayahead'][target_col_dayahead]['train'],
        all_loaders[fn]['dayahead'][target_col_dayahead]['val'],
        model_type='dayahead',
        num_epochs=50,
        lr=cfg['lr'],
        loss_fn='mae',
    )

    # Evaluate on VALIDATION set only
    da_val_preds, da_val_actuals, da_val_metrics = test_model(
        da_trained,
        all_loaders[fn]['dayahead'][target_col_dayahead]['val'],
        fold_data['target_scalers'][target_col_dayahead],
        model_type='dayahead',
    )

    # ── Real-time ──────────────────────────────────────────────────────────
    rt_model = RealTimeModel(
        input_size=INPUT_SIZE_HIST,
        n_forecast_features=INPUT_SIZE_FC,
        hidden_size=cfg['hidden_size'],
        num_layers=NUM_LAYERS,
        dropout=cfg['dropout'],
    ).to(device)

    rt_trained, rt_history = train_model(
        rt_model,
        all_loaders[fn]['realtime'][target_col_realtime]['train'],
        all_loaders[fn]['realtime'][target_col_realtime]['val'],
        model_type='realtime',
        num_epochs=50,
        lr=cfg['lr'],
        loss_fn='mae',
    )

    rt_val_preds, rt_val_actuals, rt_val_metrics = test_model(
        rt_trained,
        all_loaders[fn]['realtime'][target_col_realtime]['val'],
        fold_data['target_scalers'][target_col_realtime],
        model_type='realtime',
    )

    val_results.append({
        'config'       : cfg,
        'da_val_mae'   : da_val_metrics['mae'],
        'da_val_rmse'  : da_val_metrics['rmse'],
        'da_val_mae_pct': da_val_metrics['mae_pct'],
        'rt_val_mae'   : rt_val_metrics['mae'],
        'rt_val_rmse'  : rt_val_metrics['rmse'],
        'rt_val_mae_pct': rt_val_metrics['mae_pct'],
        'da_model'     : da_trained,
        'rt_model'     : rt_trained,
    })

# ── Print validation comparison table ─────────────────────────────────────────
print(f"\n{'='*90}")
print(f"VALIDATION RESULTS — HYPERPARAMETER COMPARISON")
print(f"{'='*90}")
print(f"{'Config':<45} {'DA MAE':>8} {'DA MAE%':>8} {'RT MAE':>8} {'RT MAE%':>8}")
print("-" * 90)

for r in val_results:
    cfg_str = str(r['config'])
    print(f"{cfg_str:<45} "
          f"{r['da_val_mae']:>8.3f} {r['da_val_mae_pct']:>7.1f}% "
          f"{r['rt_val_mae']:>8.3f} {r['rt_val_mae_pct']:>7.1f}%")

In [ ]:
# ── Select best config based on validation MAE ────────────────────────────────
# Using day-ahead MAE as the primary criterion; adjust as needed.
best = min(val_results, key=lambda r: r['da_val_mae'])
print(f"Best config: {best['config']}")
print(f"  DA val MAE : {best['da_val_mae']:.3f}  ({best['da_val_mae_pct']:.1f}%)")
print(f"  RT val MAE : {best['rt_val_mae']:.3f}  ({best['rt_val_mae_pct']:.1f}%)")

best_da_model = best['da_model']
best_rt_model = best['rt_model']

## 8. Train — Day-Ahead Model (`price day ahead`, 24 hr horizon)

In [ ]:
fold_data = folds[0]   # use the first (and typically only) fold
fn        = fold_data['fold']

da_model = DayAheadModel(
    input_size=INPUT_SIZE_HIST,
    hidden_size=HIDDEN_SIZE,
    num_layers=NUM_LAYERS,
    dropout=DROPOUT,
).to(device)

da_trained, da_history = train_model(
    da_model,
    all_loaders[fn]['dayahead'][target_col_dayahead]['train'],
    all_loaders[fn]['dayahead'][target_col_dayahead]['val'],
    model_type='dayahead',
    num_epochs=50,
    lr=1e-3,
    loss_fn='mae',
)

In [ ]:
da_preds, da_actuals, da_metrics = test_model(
    da_trained,
    all_loaders[fn]['dayahead'][target_col_dayahead]['test'],
    fold_data['target_scalers'][target_col_dayahead],
    model_type='dayahead',
)

## 9. Train — Real-Time Model (`price actual`, 1 hr horizon)

**Stream 2 inputs** (injected into dense head, not the encoder):
- `forecast solar day ahead` — published TSO forecast for target hour
- `forecast wind onshore day ahead` — published TSO forecast
- `total load forecast` — published TSO forecast
- `price day ahead` — already settled before real-time delivery

These are the same published forecasts a market participant has available at real-time inference — no data leakage.

In [ ]:
rt_model = RealTimeModel(
    input_size=INPUT_SIZE_HIST,
    n_forecast_features=INPUT_SIZE_FC,
    hidden_size=HIDDEN_SIZE,
    num_layers=NUM_LAYERS,
    dropout=DROPOUT,
).to(device)

rt_trained, rt_history = train_model(
    rt_model,
    all_loaders[fn]['realtime'][target_col_realtime]['train'],
    all_loaders[fn]['realtime'][target_col_realtime]['val'],
    model_type='realtime',
    num_epochs=50,
    lr=1e-3,
    loss_fn='mae',
)

In [ ]:
rt_preds, rt_actuals, rt_metrics = test_model(
    rt_trained,
    all_loaders[fn]['realtime'][target_col_realtime]['test'],
    fold_data['target_scalers'][target_col_realtime],
    model_type='realtime',
)

## 10. Save Trained Models

In [ ]:
os.makedirs('models', exist_ok=True)

torch.save({
    'model_state'    : da_trained.state_dict(),
    'scaler_X'       : fold_data['scaler_X'],
    'target_scaler'  : fold_data['target_scalers'][target_col_dayahead],
    'hist_feature_cols' : hist_feature_cols,
    'target_col'     : target_col_dayahead,
    'window_size'    : WINDOW_SIZE,
    'horizon'        : HORIZON_DA,
    'model_config'   : {
        'input_size'  : INPUT_SIZE_HIST,
        'hidden_size' : HIDDEN_SIZE,
        'num_layers'  : NUM_LAYERS,
        'dropout'     : DROPOUT,
    },
}, 'models/dayahead_model.pt')
print("Day-ahead model saved → models/dayahead_model.pt")

torch.save({
    'model_state'       : rt_trained.state_dict(),
    'scaler_X'          : fold_data['scaler_X'],
    'scaler_forecast'   : fold_data['scaler_forecast'],
    'target_scaler'     : fold_data['target_scalers'][target_col_realtime],
    'hist_feature_cols' : hist_feature_cols,
    'forecast_cols'     : realtime_forecast_cols,
    'target_col'        : target_col_realtime,
    'window_size'       : WINDOW_SIZE,
    'model_config'      : {
        'input_size'          : INPUT_SIZE_HIST,
        'n_forecast_features' : INPUT_SIZE_FC,
        'hidden_size'         : HIDDEN_SIZE,
        'num_layers'          : NUM_LAYERS,
        'dropout'             : DROPOUT,
    },
}, 'models/realtime_model.pt')
print("Real-time  model saved → models/realtime_model.pt")

## 11. Results Visualisation

In [ ]:
def plot_results(preds, actuals, history, model_type, target_name):
    """
    Plots for either model:
      1. Full test-period actual vs predicted
      2. First 7 days zoomed
      3. Scatter actual vs predicted
      4. Loss curves
    Day-ahead model also gets a 24hr profile plot.
    """
    is_da = (model_type == 'dayahead')

    nrows = 3 if is_da else 2
    fig   = plt.figure(figsize=(16, 5 * (nrows + 1)))
    fig.suptitle(f'{model_type.capitalize()} — {target_name}',
                 fontsize=14, fontweight='bold')
    gs = gridspec.GridSpec(nrows + 1, 2, figure=fig, hspace=0.5, wspace=0.3)

    flat_pred = preds.flatten()
    flat_act  = actuals.flatten()

    # ── Plot 1: Full test period ──────────────────────────────────────────────
    ax1 = fig.add_subplot(gs[0, :])
    ax1.plot(flat_act,  label='Actual',    alpha=0.7, color='steelblue', lw=0.8)
    ax1.plot(flat_pred, label='Predicted', alpha=0.7, color='orange',    lw=0.8)
    ax1.set_title('Full Test Period — Actual vs Predicted')
    ax1.set_xlabel('Hour index')
    ax1.set_ylabel('€/MWh')
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    # ── Plot 2: First 7 days ──────────────────────────────────────────────────
    ax2 = fig.add_subplot(gs[1, 0])
    ax2.plot(flat_act[:168],  label='Actual',    alpha=0.7, color='steelblue')
    ax2.plot(flat_pred[:168], label='Predicted', alpha=0.7, color='orange')
    ax2.set_title('First 7 Days')
    ax2.set_xlabel('Hour')
    ax2.set_ylabel('€/MWh')
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    # ── Plot 3: Scatter ───────────────────────────────────────────────────────
    ax3 = fig.add_subplot(gs[1, 1])
    ax3.scatter(flat_act, flat_pred, alpha=0.2, color='steelblue', s=4)
    lo, hi = min(flat_act.min(), flat_pred.min()), max(flat_act.max(), flat_pred.max())
    ax3.plot([lo, hi], [lo, hi], 'r--', lw=1.5, label='Perfect')
    ax3.set_title('Scatter — Actual vs Predicted')
    ax3.set_xlabel('Actual')
    ax3.set_ylabel('Predicted')
    ax3.legend()
    ax3.grid(True, alpha=0.3)

    if is_da:
        # ── Plot 4: 24hr profile (day-ahead only) ─────────────────────────────
        for j, (sample_idx, title) in enumerate([(0, 'Sample Day 1'), (100, 'Sample Day 100')]):
            ax = fig.add_subplot(gs[2, j])
            ax.plot(range(24), actuals[sample_idx], label='Actual',
                    color='steelblue', marker='o', markersize=4)
            ax.plot(range(24), preds[sample_idx],   label='Predicted',
                    color='orange',    marker='x', markersize=4)
            ax.set_title(f'24hr Profile — {title}')
            ax.set_xlabel('Hour of day')
            ax.set_ylabel('€/MWh')
            ax.legend()
            ax.grid(True, alpha=0.3)

    # ── Loss curves ───────────────────────────────────────────────────────────
    ax_loss = fig.add_subplot(gs[nrows, :])
    epochs  = range(1, len(history['train_loss']) + 1)
    ax_loss.plot(epochs, history['train_loss'], label='Train', color='steelblue', lw=1.5)
    ax_loss.plot(epochs, history['val_loss'],   label='Val',   color='orange',    lw=1.5)
    ax_loss.set_title('Loss Curves')
    ax_loss.set_xlabel('Epoch')
    ax_loss.set_ylabel('Loss')
    ax_loss.legend()
    ax_loss.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

In [ ]:
plot_results(da_preds, da_actuals, da_history,
             model_type='dayahead', target_name=target_col_dayahead)

plot_results(rt_preds, rt_actuals, rt_history,
             model_type='realtime', target_name=target_col_realtime)

In [ ]:
print(f"\n{'='*60}")
print(f"RESULTS SUMMARY — Fold {fn}")
print(f"{'='*60}")
print(f"\n{'Model':<20} {'Target':<22} {'MAE':>8} {'RMSE':>8} {'MAE%':>7}")
print("-" * 70)

for label, metrics, target in [
    ('Day-Ahead (24hr)', da_metrics, target_col_dayahead),
    ('Real-Time (1hr)',  rt_metrics, target_col_realtime),
]:
    status = '✅' if metrics['mae_pct'] < 10 else ('⚠️' if metrics['mae_pct'] < 20 else '❌')
    print(f"{label:<20} {target:<22} "
          f"{metrics['mae']:>8.3f} {metrics['rmse']:>8.3f} "
          f"{metrics['mae_pct']:>6.1f}%  {status}")

## Test Models

In [ ]:
# ── Final evaluation on TEST set ──────────────────────────────────────────────
# Run ONCE only, after hyperparameters are fully locked.
# Do not go back and adjust anything after seeing these numbers.

da_test_preds, da_test_actuals, da_test_metrics = test_model(
    best_da_model,
    all_loaders[fn]['dayahead'][target_col_dayahead]['test'],
    fold_data['target_scalers'][target_col_dayahead],
    model_type='dayahead',
)

rt_test_preds, rt_test_actuals, rt_test_metrics = test_model(
    best_rt_model,
    all_loaders[fn]['realtime'][target_col_realtime]['test'],
    fold_data['target_scalers'][target_col_realtime],
    model_type='realtime',
)

print(f"\n{'='*60}")
print(f"FINAL TEST RESULTS")
print(f"{'='*60}")
print(f"\n{'Model':<20} {'MAE':>8} {'RMSE':>8} {'MAE%':>7}")
print("-" * 50)
for label, metrics in [
    ('Day-Ahead (24hr)', da_test_metrics),
    ('Real-Time (1hr)',  rt_test_metrics),
]:
    status = '✅' if metrics['mae_pct'] < 10 else ('⚠️' if metrics['mae_pct'] < 20 else '❌')
    print(f"{label:<20} {metrics['mae']:>8.3f} {metrics['rmse']:>8.3f} "
          f"{metrics['mae_pct']:>6.1f}%  {status}")

## 12. Inference Functions - NOT NEEDED (PRODUCTION GRADE)

Both functions expect **pre-scaled** inputs (apply the saved scalers first).

In [ ]:
def predict_dayahead(model, last_168h_scaled, target_scaler, device):
    """
    Predict the next 24 hourly day-ahead prices.

    Parameters
    ----------
    last_168h_scaled : np.ndarray, shape (168, n_hist_features)
        Last 168 hours of historical features, already scaled with scaler_X.
    target_scaler    : fitted MinMaxScaler for 'price day ahead'
    device           : torch.device

    Returns
    -------
    pd.Series — 24 predicted €/MWh values indexed by UTC hour
    """
    model.eval()
    with torch.no_grad():
        x = torch.FloatTensor(last_168h_scaled).unsqueeze(0).to(device)
        pred_scaled, _ = model(x)                            # (1, 24)
        pred_orig = target_scaler.inverse_transform(
            pred_scaled.cpu().numpy().reshape(-1, 1)
        ).flatten()

    tomorrow = pd.Timestamp.utcnow().normalize() + pd.Timedelta(days=1)
    hours    = pd.date_range(tomorrow, periods=24, freq='h')
    return pd.Series(pred_orig, index=hours, name='price_day_ahead_eur_mwh')


def predict_realtime(model, last_168h_scaled, forecast_t_scaled,
                     target_scaler, device):
    """
    Predict the real-time price for the next single hour.

    Parameters
    ----------
    last_168h_scaled   : np.ndarray, shape (168, n_hist_features)
        Last 168 hours of historical features, already scaled with scaler_X.
    forecast_t_scaled  : np.ndarray, shape (n_forecast_features,)
        Published forecast features for the target hour, scaled with
        scaler_forecast.
        Order: [forecast_solar_day_ahead, forecast_wind_onshore_day_ahead,
                total_load_forecast, price_day_ahead]
    target_scaler      : fitted MinMaxScaler for 'price actual'
    device             : torch.device

    Returns
    -------
    float — predicted €/MWh for the next hour
    """
    model.eval()
    with torch.no_grad():
        x_h  = torch.FloatTensor(last_168h_scaled).unsqueeze(0).to(device)
        x_fc = torch.FloatTensor(forecast_t_scaled).unsqueeze(0).to(device)
        pred_scaled, _ = model(x_h, x_fc)                   # (1, 1)
        pred_orig = target_scaler.inverse_transform(
            pred_scaled.cpu().numpy().reshape(-1, 1)
        ).flatten()[0]

    return float(pred_orig)

### Example: Running Inference on a Test Sample

In [ ]:
# ── Day-ahead example ─────────────────────────────────────────────────────────
# Take the last 168 hours of the test set as a stand-in for live data
X_test_da   = fold_data['X_test']
sample_hist = X_test_da[-168:]                  # (168, n_hist_features)

da_series = predict_dayahead(
    da_trained, sample_hist,
    fold_data['target_scalers'][target_col_dayahead],
    device,
)
print("Day-Ahead Predictions (next 24 hours):")
print(da_series.round(2).to_string())

# ── Real-time example ─────────────────────────────────────────────────────────
# Take the last 168 hours of history + the forecast for the very next hour
X_test_rt   = fold_data['X_test']
Xfc_test_rt = fold_data['Xfc_test']

sample_hist_rt = X_test_rt[-168:]              # (168, n_hist_features)
sample_fc_t    = Xfc_test_rt[-1]               # (n_forecast_features,)

rt_price = predict_realtime(
    rt_trained, sample_hist_rt, sample_fc_t,
    fold_data['target_scalers'][target_col_realtime],
    device,
)
print(f"\nReal-Time Prediction (next hour): {rt_price:.2f} €/MWh")